# 04 · Safety & Hallucination Mitigation

> **Source notes:** `SafetyAndHallucination.md`

A system that is right 95% of the time is dangerous if users can't tell which 5% is wrong. This notebook demonstrates the layered mitigation stack:

- **Hallucination taxonomy** — generating examples of each hallucination type
- **Grounding constraints** — prompt-level mitigation
- **NLI-based claim verification** — post-generation check using a cross-encoder
- **Self-consistency sampling** — flag low-agreement answers
- **Confidence elicitation** — ask the model to rate its own certainty

All LLM calls use **Ollama** (local, no API key needed).

## 0 · Environment Setup

```bash
ollama pull phi3:mini
ollama serve
```

In [ ]:
def chat(system: str, user: str, temperature: float = 0.0, max_tokens: int = 200):
    """
    TODO #1: Implement `chat()`.

    Steps:
    1. Call `run()` to produce the result
    2. Process data
    3. Compute `MODEL`
    4. Define helper function `chat()`
    5. Process data

    Hint:
    resp = ollama.chat(???)

    Returns: resp["message"]["content"].strip()
    """
    raise NotImplementedError("TODO: implement chat()")

## 1 · Hallucination Taxonomy — Generating Examples

| Type | Description |
|---|---|
| **Factual hallucination** | Plausible-sounding but wrong fact |
| **Confabulation** | Fabricated citation with correct-looking format |
| **Sycophantic hallucination** | Agreeing with a false premise |
| **Specification overreach** | Adding unrequested content beyond constraints |

We provoke each type to understand what we're defending against.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `ans` using `hallucination()`
# 2. Compute `ans` using `overreach()`
#
# Hint:
#    # implement using the APIs described above

## 2 · Prompt-Level Mitigation — Grounding Constraints

The single most effective prompt-level mitigation:

```
"Base your answer ONLY on the provided context.
 If the answer is not present, say: 'I don't have that information.'"
```

We compare an ungrounded prompt vs a grounded prompt with the same out-of-context query.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `MENU_CONTEXT` using `99()`
# 2. Compute `query`
# 3. Compute `ungrounded_system`
# 4. Compute `user_msg`
# 5. Call `fill()` to produce the result
#
# Hint:
#    # implement using the APIs described above

## 3 · NLI-Based Claim Verification

After generation, verify each claim in the output is **entailed** by the source context using a Natural Language Inference (NLI) cross-encoder model.

Label mapping:
- **ENTAILMENT** → claim is supported by context
- **NEUTRAL** → claim may or may not be true based on context
- **CONTRADICTION** → claim is contradicted by context

We use `cross-encoder/nli-deberta-v3-small` (runs locally, ~150 MB).

In [ ]:
def verify_claim(claim: str, context: str):
    """
    TODO #4: Implement `verify_claim()`.

    Steps:
    1. Process data
    2. Compute `nli_model` using `model()`
    3. Define helper function `verify_claim()`
    4. Compute `claims`
    5. Process data

    Hint:
    nli_model = CrossEncoder(???)
    scores = nli_model.predict(???)
    nli_model.predict(???)

    Returns: {"label": label, "confidence": round(...
    """
    raise NotImplementedError("TODO: implement verify_claim()")

## 4 · Self-Consistency Sampling — Detecting Uncertain Answers

Generate **N answers at temperature > 0**, then take the majority vote. Low-agreement answers (no clear majority) are flagged for human review.

This is useful for high-stakes queries where a single-pass answer may be wrong.

In [ ]:
def self_consistency(question: str, n: int = 5, temperature: float = 0.7):
    """
    TODO #5: Implement `self_consistency()`.

    Steps:
    1. Define helper function `self_consistency()`
    2. Call `vote()` to produce the result
    3. Compute `q1` using `question()`
    4. Process data

    Hint:
    vote = Counter(???)
    resp = ollama.chat(???)

    Returns: {"answers": answers, "majority": vote...
    """
    raise NotImplementedError("TODO: implement self_consistency()")

## 5 · Confidence Elicitation

Ask the model to rate its own confidence. Low-confidence answers trigger a fallback (human escalation or "I don't know" response). Note: self-reported confidence is not perfectly calibrated, but it correlates with actual accuracy enough to be useful.

In [ ]:
def elicit_confidence(question: str):
    """
    TODO #6: Implement `elicit_confidence()`.

    Steps:
    1. Compute `conf_system`
    2. Process data
    3. Compute `questions`
    4. Process data
    5. Define helper function `elicit_confidence()`
    6. Call `split()` to produce the result

    Hint:
    match = re.search(???)
    confidence = match.group(???)
    answer_text = raw.split(???)

    Returns: {"answer": answer_text, "confidence":...
    """
    raise NotImplementedError("TODO: implement elicit_confidence()")

## Summary — Layered Mitigation Stack

| Layer | Technique | This Notebook |
|---|---|---|
| Prompt-level | Grounding constraint | Section 2 |
| Post-generation | NLI claim verification | Section 3 |
| Post-generation | Self-consistency sampling | Section 4 |
| Post-generation | Confidence elicitation | Section 5 |

Apply all four layers in production for high-stakes outputs. No single technique eliminates hallucination — defence in depth is the only reliable strategy.

**Next:** [EvaluatingAISystems/notebook.ipynb](../EvaluatingAISystems/notebook.ipynb) — systematic evaluation beyond ad-hoc testing.